In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

X = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv', index_col='id') 
X_test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv', index_col='id')

X.dropna(axis=0, subset=['Will_Buy_EV'], inplace=True)

y = X['Will_Buy_EV'].map({'Yes': 1, 'No': 0})

X.drop(['Will_Buy_EV'], axis=1, inplace=True)

X.head()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Identify categorical and numerical columns directly from X
categorical_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
numerical_cols = X.select_dtypes(exclude=['object', 'bool']).columns.tolist()

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# Feature engineering
## Added variables
- Total charging stations (sum of at home and work)
- Income per car owned
- Commute km relative to income
- Environment concerns amplified possibilities by income

Accuracy stayed the same.

In [ ]:
def add_features(df):
    df = df.copy()
    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['Income_per_Car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1)
    df['Commute_Income_Ratio'] = df['Daily_Commute_km'] / (df['Annual_Income_USD'] + 1) * 10000
    df['Environment_Incom_Multiplication'] = df['Environmental_Concern_Level'] * (df['Annual_Income_USD'] + 1)
    return df

X = add_features(X)
X_test = add_features(X_test)

Results without XGBoost: 

```
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best CV Accuracy: 0.8949690519096359
Best Parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}
```

## XGBoost

Results with XGBoost:
```
Fitting 3 folds for each of 63 candidates, totalling 189 fits
Best CV Accuracy: 0.8984409239938813
Best Parameters: {'model__learning_rate': 0.03, 'model__max_depth': 6, 'model__n_estimators': 800}
```

# Ensembling
## Add LightGBM

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

# best_xgb_model = grid_search.best_estimator_

best_xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        random_state=0,
        n_jobs=2,
        eval_metric='logloss'
    ))
])


lgbm_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    random_state=0,
    n_jobs=2
)

lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', lgbm_model)
])

# Combine XGBoost and LightGBM
ensemble_model = VotingClassifier(
    estimators=[
        ('xgb', best_xgb_model),
        ('lgb', lgbm_pipeline)
    ],
    voting='soft'
)

cv_scores = cross_val_score(ensemble_model, X, y, cv=3, scoring='accuracy', n_jobs=-1)
print(f"Ensemble CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

ensemble_model.fit(X, y)

ensemble_predictions = ensemble_model.predict(X_test)
submission_ensemble = pd.DataFrame({
    'id': X_test.index,
    'Will_Buy_EV': ensemble_predictions
})
submission_ensemble.to_csv('submission_ensemble.csv', index=False)
print("Saved Ensemble Submission")

## Add CatBoost

In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.03,
    depth=6,
    random_seed=0,
    verbose=0,
    thread_count=2
)

cat_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', cat_model)
])

ensemble_model_3 = VotingClassifier(
    estimators=[
        ('xgb', best_xgb_model),
        ('lgb', lgbm_pipeline),
        ('cat', cat_pipeline)
    ],
    voting='soft',
    weights=[1.2, 1.0, 1.0]
)

ensemble_model_3.fit(X, y)

chunks = np.array_split(X_test, 5)
predictions_list_3 = []

for i, chunk in enumerate(chunks):
    chunk_pred = ensemble_model_3.predict(chunk)
    predictions_list_3.append(chunk_pred)

ensemble_predictions_3 = np.concatenate(predictions_list_3)

submission_ensemble_3 = pd.DataFrame({
    'id': X_test.index,
    'Will_Buy_EV': ensemble_predictions_3
})
submission_ensemble_3.to_csv('submission_ensemble_3_models.csv', index=False)

# Optuna Tuning

## Optuna Results 
During an Optuna Tuning Session, the parameters converged to: 
- Best Trial: Trial 280
- Best Score: 0.8975271698679395

### Best-Parameters
```
n_estimators: 774
learning_rate: 0.023002832746440098
max_depth: 4
min_child_weight: 3
subsample: 0.8726630838164714
colsample_bytree: 0.7333028878897986
```

## Using Optuna Results for model, further changes
I noticed very bad results in the actual test data. Hence I made those changes:
- Dynamic Class Weight Calculation: Added logic to calculate the exact ratio of the majority ("No") to minority ("Yes") class to address the data imbalance.
- Model Parameter Updates:
    - Added `scale_pos_weight=scale_weight` to both the XGBoost and LightGBM pipelines.
    - Added `auto_class_weights='Balanced'` to the CatBoost pipeline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Ensure y is defined by running earlier cells
num_negative = (y == 0).sum()
num_positive = (y == 1).sum()
scale_weight = num_negative / num_positive
print(f"Calculated scale_pos_weight: {scale_weight:.4f}")

best_xgb_params = {
    'n_estimators': 774,
    'learning_rate': 0.023002832746440098,
    'max_depth': 4,
    'min_child_weight': 3,
    'subsample': 0.8726630838164714,
    'colsample_bytree': 0.7333028878897986,
    'scale_pos_weight': scale_weight
}

optuna_xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(**best_xgb_params, random_state=0, n_jobs=2, eval_metric='logloss'))
])

lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        scale_pos_weight=scale_weight,
        random_state=0,
        n_jobs=2
    ))
])

cat_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', CatBoostClassifier(
        iterations=800,
        learning_rate=0.03,
        depth=6,
        auto_class_weights='Balanced',
        random_seed=0,
        verbose=0,
        thread_count=2
    ))
])

final_ensemble = VotingClassifier(
    estimators=[
        ('xgb', optuna_xgb_pipeline),
        ('lgb', lgbm_pipeline),
        ('cat', cat_pipeline)
    ],
    voting='soft',
    weights=[1.2, 1.0, 1.0] 
)

print("Running Cross-Validation on final ensemble...")
cv_scores = cross_val_score(final_ensemble, X, y, cv=3, scoring='accuracy', n_jobs=-1)

print("\nFinal Model Statistics:")
print(f"- Cross-Validation Accuracy Scores: {cv_scores}")
print(f"- Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"- Standard Deviation: {cv_scores.std():.4f}\n")

print("Training final ensemble on full data...")
final_ensemble.fit(X, y)

chunks = np.array_split(X_test, 5)
predictions_list = []

for i, chunk in enumerate(chunks):
    predictions_list.append(final_ensemble.predict(chunk))

test_predictions = np.concatenate(predictions_list)

submission = pd.DataFrame({
    'id': X_test.index,
    'Will_Buy_EV': test_predictions
})

submission.to_csv('submission_optuna_ensemble_weighted.csv', index=False)
print("Saved Optuna Submission: submission_optuna_ensemble_weighted.csv")

# Final Results

Dieses finale Voting-Ensemble (XGBoost, LightGBM, CatBoost) nutzt Optuna-optimierte Hyperparameter und integriert dynamische Klassengewichte (`scale_pos_weight` & `auto_class_weights='Balanced'`). Dies steuert der stark unbalancierten Zielvariable (Verhältnis ca. 1:4.7) effektiv entgegen und verhindert, dass das Modell blind die Mehrheitsklasse vorhersagt.

### Kaggle Leaderboard Performance:
- **Private Score:** 0.87649
- **Public Score:** 0.87592